In [0]:
#How to Identify Whether a File Is Binary
#Method A — Try reading as text (simple & common)
def is_binary(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            f.read()
        return False
    except UnicodeDecodeError:
        return True
"""
Logic

Text files → decode successfully

Binary files → raise UnicodeDecodeError

Use case

ETL pipeline deciding:

parse as CSV/text

or treat as binary (image, parquet, gzip)
"""

#Method B — Check for NULL bytes (faster heuristic)
def is_binary_fast(file_path):
    with open(file_path, "rb") as f:
        chunk = f.read(1024)
    return b"\x00" in chunk
"""
Logic

Many binary files contain NULL byte (\x00)

Text files usually don’t

Used in

File upload validation

Data ingestion services

# When to Use bytes, bytearray, and memoryview
A. Use bytes → Read-only binary data
Scenario

Reading:

images

PDFs

parquet

compressed files
"""
with open("image.png", "rb") as f:
    data = f.read()   # bytes
"""
Why?

Safe

immutable

memory efficient

Most common in data pipelines.
"""
#B. Use bytearray → You need to MODIFY binary data
Scenario 1 — Fix corrupted header
with open("file.bin", "rb") as f:
    data = bytearray(f.read())

data[0] = 255  # modify byte

with open("fixed.bin", "wb") as f:
    f.write(data)
"""
Scenario 2 — Encryption / encoding

Change bytes during processing

Patch values before saving

Key idea:
Choose bytearray only when mutation is required.
"""
# C. Use memoryview → Large data & performance critical
# Scenario 1 — Process huge file without copying
with open("large.bin", "rb") as f:
    data = f.read()

view = memoryview(data)
chunk = view[0:1024]   # no copy
"""
Scenario 2 — Streaming / zero-copy pipelines

Used in:

high-performance networking

big data ingestion

ML preprocessing

cloud upload buffers"""


"""
Production-Safe File Handling Utilities (Python)
1️⃣ Safe Text Read

✔ Handles encoding
✔ Prevents crash with clear error
"""
def read_text(file_path: str) -> str:
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        raise RuntimeError(f"File not found: {file_path}")
    except UnicodeDecodeError:
        raise RuntimeError(f"Encoding error while reading: {file_path}")

"""
Used in PROD for:

configs

logs

JSON/CSV raw text
"""
2️⃣ Safe Text Write (Atomic-style overwrite)
def write_text(file_path: str, content: str) -> None:
    try:
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(content)
    except OSError as e:
        raise RuntimeError(f"Failed writing file {file_path}: {e}")

"""
Used in PROD for:

reports

pipeline outputs

checkpoints

3️⃣ Safe Append (Logging Pattern)"""
def append_text(file_path: str, content: str) -> None:
    try:
        with open(file_path, "a", encoding="utf-8") as f:
            f.write(content + "\n")
    except OSError as e:
        raise RuntimeError(f"Failed appending file {file_path}: {e}")

"""
Critical in:

ETL logs

audit trails

monitoring

4️⃣ Memory-Safe Line Iterator (for Large Files)

✔ Never loads full file into RAM
"""
def iter_lines(file_path: str):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                yield line.rstrip("\n")
    except OSError as e:
        raise RuntimeError(f"Failed reading lines from {file_path}: {e}")

"""
Top real-world usage:

log processing

streaming ETL

big text datasets

5️⃣ Safe Binary Read
"""
def read_binary(file_path: str) -> bytes:
    try:
        with open(file_path, "rb") as f:
            return f.read()
    except OSError as e:
        raise RuntimeError(f"Binary read failed for {file_path}: {e}")

"""
Used in PROD for:

images

PDFs

parquet/gzip

ML artifacts

6️⃣ Chunked Binary Reader (Most Important for Data Engineering)

✔ Prevents memory crash on GB/TB files
"""
def iter_binary_chunks(file_path: str, size: int = 1024 * 1024):
    try:
        with open(file_path, "rb") as f:
            while True:
                chunk = f.read(size)
                if not chunk:
                    break
                yield chunk
    except OSError as e:
        raise RuntimeError(f"Chunk read failed for {file_path}: {e}")

"""
REAL PROD USE:

cloud uploads

streaming ingestion

huge datasets

👉 This is one of the most important functions in data engineering.

7️⃣ File Existence Check (Fail-Fast Guard)
"""
from pathlib import Path

def ensure_exists(file_path: str) -> None:
    if not Path(file_path).exists():
        raise RuntimeError(f"Required file missing: {file_path}")

"""
Used everywhere in PROD pipelines
to avoid silent failures.

Why These Are Production-Grade

They follow real engineering principles:

✔ use with open() → auto close, no leaks
✔ explicit encoding → avoids hidden bugs
✔ controlled exceptions → no silent crashes
✔ memory-safe iteration → handles large data
✔ minimal surface area → easy to test & reuse

The True 20/80 Insight for Production

If you master only these 7 utilities, you can safely handle:

ETL pipelines

log processors

ML data loaders

backend services

automation jobs

👉 That’s most real-world file handling in production."""

"""

Safe Text Read
def read_text(file_path: str) -> str:
Defines a function named read_text.
file_path: str → expects a string path.
-> str → guarantees the function returns text.
👉 Production benefit: type clarity + easier debugging.
    try:
Starts protected block.
👉 Prevents pipeline crash.
        with open(file_path, "r", encoding="utf-8") as f:
"r" → read mode.
encoding="utf-8" → avoids hidden encoding bugs.
with → auto-closes file (very important in production).
            return f.read()
Reads entire file into memory.
👉 Fast for small/medium files only.
    except FileNotFoundError:
        raise RuntimeError(f"File not found: {file_path}")
Converts low-level error → clear business error.
👉 Useful in ETL logs & monitoring.
    except UnicodeDecodeError:
        raise RuntimeError(f"Encoding error while reading: {file_path}")
Handles wrong encoding safely.
✅ When to use
Use ONLY IF:
File size is small (<10–50 MB)
You need entire content at once
Example:
config.json
SQL query file
small CSV
2️⃣ Safe Text Write
def write_text(file_path: str, content: str) -> None:
Writes text to disk.
Returns nothing.
    try:
        with open(file_path, "w", encoding="utf-8") as f:
"w" → overwrite file completely.
👉 Important production behavior.
            f.write(content)
Writes full string.
    except OSError as e:
        raise RuntimeError(f"Failed writing file {file_path}: {e}")
Handles disk permission / space errors.
✅ When to use
Generating reports
Saving processed ETL output
Writing checkpoint/state files
Never use for logs → use append instead.
3️⃣ Safe Append (Logging Pattern)
def append_text(file_path: str, content: str) -> None:
Adds new line without deleting old data.
        with open(file_path, "a", encoding="utf-8") as f:
"a" → append mode.
👉 Critical for audit safety.
            f.write(content + "\n")
Ensures each log entry is new line.
✅ When to use
Real production uses:
ETL run logs
error tracking
audit history
👉 Most common real-world file operation.
4️⃣ Memory-Safe Line Iterator
def iter_lines(file_path: str):
Returns generator, not full file.
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                yield line.rstrip("\n")
Key idea:
Reads one line at a time
Uses constant memory
👉 Works for GB-size logs.
    except OSError as e:
Handles failures safely.
✅ When to use
This is HUGE in data engineering.
Use for:
log processing
large CSV scanning
streaming ETL
reading millions of rows
👉 Default choice for large text.
5️⃣ Safe Binary Read
def read_binary(file_path: str) -> bytes:
Returns raw bytes.
        with open(file_path, "rb") as f:
            return f.read()
"rb" → binary mode.
Reads full file.
✅ When to use
For non-text files:
images
PDFs
parquet
ML models
⚠ Only if file fits in memory.
6️⃣ Chunked Binary Reader (MOST IMPORTANT)
def iter_binary_chunks(file_path: str, size: int = 1024 * 1024):
Reads 1 MB at a time by default.
        with open(file_path, "rb") as f:
            while True:
                chunk = f.read(size)
                if not chunk:
                    break
                yield chunk
Critical production concept:
Never loads full file
Safe for GB/TB data
Used in:
cloud upload
streaming ingestion
distributed systems
👉 Top 3 most important real-world patterns.
7️⃣ Ensure File Exists (Fail-Fast)
from pathlib import Path
Modern, safer than os.
def ensure_exists(file_path: str) -> None:
    if not Path(file_path).exists():
        raise RuntimeError(f"Required file missing: {file_path}")
Why this matters:
Prevents:
silent ETL failures
partial pipelines
bad downstream data
👉 Used in every mature production pipeline.


"""